# Reviewed custom labels and a locked mixed-domain development test

The human-reviewed workbook supplies simple violation/no_violation targets and
16 custom point timestamps. The fixed study uses 40 original recordings and 18
reviewed MOV recordings: train34 / validation12 / test12. The MOV family had
the largest usable label coverage before model scores. Every custom MP4 is
excluded from training, validation, test and model selection.

Known native source overlaps stay together. These groups are not human-verified
physical scene identities. Custom footage comes from one filming batch, and the
original recordings were examined during earlier development. This camera-exclusive
within-session test therefore does not establish unseen-scene/session accuracy.

Frozen visual features are computed locally before upload. Training runs in native
Python with no audio, hosted LLM APIs, private descriptions or video uploads to AI
services. More than 80% and at least 87% are goals, not assumed outcomes.


In [ ]:
# Restore exact source bytes; reuse an existing original feature cache in this workspace.
import base64, hashlib, io, json, os, sys, zipfile
from pathlib import Path
BASE = Path('/content') if Path('/content').is_dir() else Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORKSPACE = BASE / 'datathon_reviewed_workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)
SOURCE_ZIP = base64.b64decode('...')
assert hashlib.sha256(SOURCE_ZIP).hexdigest() == '945d2cb9a9e42faddcd9e77b9297f934f25b888536d396b23f6092b99f6de32d', 'Embedded source checksum differs'
with zipfile.ZipFile(io.BytesIO(SOURCE_ZIP)) as archive:
    seen = set()
    for member in archive.infolist():
        target = (WORKSPACE / member.filename).resolve()
        if not target.is_relative_to(WORKSPACE.resolve()) or target in seen:
            raise ValueError('Unsafe embedded source path')
        seen.add(target)
    manifest = json.loads(archive.read('SOURCE_MANIFEST.json'))
    for name, digest in manifest['members_sha256'].items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == digest, 'Source member checksum differs'
    archive.extractall(WORKSPACE)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
print('Workspace:', WORKSPACE)
print('Source SHA-256:', hashlib.sha256(SOURCE_ZIP).hexdigest())


In [ ]:
# Public packages only; this notebook performs no video/LLM API calls.
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements-ml.txt', 'catboost==1.2.10'], cwd=WORKSPACE, check=True)
import torch, catboost
assert catboost.__version__ == '1.2.10'
print('Torch:', torch.__version__, 'CatBoost:', catboost.__version__, 'CPU heads')


## Restore private numeric inputs
Only checksummed train/validation caches are extracted initially. Test caches stay physically unavailable to the fitting command.

In [ ]:
# Upload the private reviewed_numeric_dataset.zip in Colab's Files pane.
# Only train/validation NPZs are materialized before the winner is frozen.
DATASET_ZIP = next((candidate for candidate in (
    BASE/'reviewed_numeric_dataset.zip', WORKSPACE/'reviewed_numeric_dataset.zip')
    if candidate.is_file()), None)
assert DATASET_ZIP is not None, 'Upload the separate private numeric dataset ZIP first.'
with zipfile.ZipFile(DATASET_ZIP) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), 'Duplicate dataset archive members'
    inventory = json.loads(archive.read('DATASET_MANIFEST.json'))
    checksums = inventory['members_sha256']
    assert set(names) == set(checksums) | {'DATASET_MANIFEST.json'}
    for name in checksums:
        assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
        assert name in {'dataset/manifest.json', 'dataset/split.json'} or (
            len(Path(name).parts) == 3 and Path(name).parts[:2] in
            {('features', 'raw_clip'), ('features', 'semantic18')} and Path(name).suffix == '.npz')
    def read_verified_member(name):
        payload = archive.read(name)
        assert hashlib.sha256(payload).hexdigest() == checksums[name], 'Dataset member checksum mismatch'
        return payload
    records = json.loads(read_verified_member('dataset/manifest.json'))
    split = json.loads(read_verified_member('dataset/split.json'))
    all_keys = {row['sha256'] for row in records}
    assert len(records) == len(all_keys) == 58
    assert set(split['train']) | set(split['validation']) | set(split['test']) == all_keys
    assert [len(split[name]) for name in ('train', 'validation', 'test')] == [34, 12, 12]
    assert split['protocol']['type'] == 'camera_exclusive_source_overlap_development'
    assert split['protocol']['opposite_camera_excluded_globally'] is True
    expected_members = {'dataset/manifest.json', 'dataset/split.json'} | {
        f'features/{representation}/{key}.npz' for representation in ('raw_clip', 'semantic18') for key in all_keys}
    assert set(checksums) == expected_members
    for name in ('dataset/manifest.json', 'dataset/split.json'):
        target = WORKSPACE/name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(read_verified_member(name))
    for representation in ('raw_clip', 'semantic18'):
        for key in sorted(set(split['train']) | set(split['validation'])):
            name = f'features/{representation}/{key}.npz'
            target = WORKSPACE/name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(read_verified_member(name))
    assert all(not (WORKSPACE/f'features/{representation}/{key}.npz').exists()
        for representation in ('raw_clip', 'semantic18') for key in split['test']), (
        'Test caches already exist: use a fresh workspace for this experiment.')
RUN = WORKSPACE/'runs/reviewed_v1'
RUN.mkdir(parents=True, exist_ok=True)
(RUN/'dataset_archive_receipt.json').write_text(json.dumps({
    'dataset_zip_sha256': hashlib.sha256(DATASET_ZIP.read_bytes()).hexdigest(),
    'members_sha256_count': len(checksums), 'records': len(records),
    'train': len(split['train']), 'validation': len(split['validation']), 'test': len(split['test']),
    'test_feature_files_materialized_before_selection': sum(
        (WORKSPACE/f'features/{representation}/{key}.npz').exists()
        for representation in ('raw_clip', 'semantic18') for key in split['test'])}, indent=2))
print('Dataset restored: 58 recordings, train34 / validation12 / test12.')
print('Test feature files physically withheld until the selection lock exists.')


In [ ]:
# Fresh processes enforce an offline socket guard and bounded run times.
from tools.run_scene_temporal_colab import run_guarded
COMMON = ['--manifest', 'dataset/manifest.json', '--split', 'dataset/split.json',
    '--originals-from', 'reference/clip_baseline_cv.json',
    '--feature-set', 'raw_clip=features/raw_clip',
    '--feature-set', 'semantic18=features/semantic18', '--output-dir', RUN]
run_guarded(WORKSPACE, RUN, 'focused_tests', 'unittest',
    ['discover', '-s', 'tests', '-p', 'test_reviewed_models.py', '-v'], 600)
RUN_FULL_TESTS = False
if RUN_FULL_TESTS:
    run_guarded(WORKSPACE, RUN, 'full_tests', 'unittest',
        ['discover', '-s', 'tests', '-v'], 900)


## Fit and freeze the validation-selected winner
The fixed candidate list uses training data for preprocessing and fitted heads. Validation selects the representation, candidate and threshold. Test features remain unopened.

In [ ]:
# Classifiers, representation and threshold are selected on validation only.
assert not (RUN/'LOCKED_TEST_USED.json').exists(), 'This run already opened its test.'
run_guarded(WORKSPACE, RUN, 'fit', 'reviewed_experiment', ['fit', *COMMON,
    '--threads', '2', '--selection-metric', 'accuracy'], 1800)
selection = json.loads((RUN/'selection.json').read_text())
lock = json.loads((RUN/'selection_lock.json').read_text())
assert selection['test_feature_files_opened'] == 0
assert hashlib.sha256((RUN/'model_reviewed.joblib').read_bytes()).hexdigest() == lock['model_sha256']
assert hashlib.sha256((RUN/'selection.json').read_bytes()).hexdigest() == lock['selection_sha256']
print('Frozen validation-selected winner:', selection['winner']['candidate'])
print('Validation metrics:', selection['winner']['validation_metrics'])
print('Test feature files opened during fitting:', selection['test_feature_files_opened'])


## Score the locked test once
The model and selection hashes must match the frozen lock before the winner's test caches are extracted. The scoring command refuses to reopen a used test.

In [ ]:
# Open only the frozen winner's test representation, once, after selection.
from datetime import datetime, timezone
lock = json.loads((RUN/'selection_lock.json').read_text())
assert hashlib.sha256((RUN/'model_reviewed.joblib').read_bytes()).hexdigest() == lock['model_sha256']
assert hashlib.sha256((RUN/'selection.json').read_bytes()).hexdigest() == lock['selection_sha256']
assert not (RUN/'LOCKED_TEST_USED.json').exists(), 'The locked test cannot be reopened.'
archive_sha256 = hashlib.sha256(DATASET_ZIP.read_bytes()).hexdigest()
archive_receipt = json.loads((RUN/'dataset_archive_receipt.json').read_text())
assert archive_sha256 == archive_receipt['dataset_zip_sha256'], 'Dataset archive changed after restoration'
assert archive_receipt['test_feature_files_materialized_before_selection'] == 0
winner_representation = selection['winner']['candidate']['representation']
with zipfile.ZipFile(DATASET_ZIP) as archive:
    checksums = json.loads(archive.read('DATASET_MANIFEST.json'))['members_sha256']
    for key in split['test']:
        name = f'features/{winner_representation}/{key}.npz'
        payload = archive.read(name)
        assert hashlib.sha256(payload).hexdigest() == checksums[name]
        (WORKSPACE/name).write_bytes(payload)
(RUN/'test_feature_release.json').write_text(json.dumps({
    'model_sha256': lock['model_sha256'],
    'selection_lock_sha256': hashlib.sha256((RUN/'selection_lock.json').read_bytes()).hexdigest(),
    'archive_sha256': archive_sha256, 'released_test_sha256': sorted(split['test']),
    'representation': winner_representation,
    'released_at': datetime.now(timezone.utc).isoformat(), 'completed': True}, indent=2))
run_guarded(WORKSPACE, RUN, 'score', 'reviewed_experiment', ['score', *COMMON], 1800)
report = json.loads((RUN/'locked_test_report.json').read_text())
print('Locked mixed-domain development test metrics:')
print(json.dumps(report['metrics'], indent=2))
print('Per-domain metrics:', json.dumps(report['domain_metrics'], indent=2))
print('Development accuracy >80%:', report['metrics']['accuracy'] > .80)
print('Development accuracy >=87%:', report['metrics']['accuracy'] >= .87)
print('This within-session development test does not establish new-scene/session accuracy.')


In [ ]:
# Cached-feature inference receives no verdict or event timestamp inputs.
representative = sorted(split['train'])[0]
run_guarded(WORKSPACE, RUN, 'inference', 'reviewed_experiment', ['predict',
    '--feature', WORKSPACE/f'features/{winner_representation}/{representative}.npz',
    '--model', RUN/'model_reviewed.joblib', '--output-dir', RUN/'inference'], 120)
print('Prediction verification covers an existing feature cache; it is not uncached video latency.')


In [ ]:
# Recompute metrics and verify split ownership, model lock and release chronology.
run_guarded(WORKSPACE, RUN, 'summary', 'tools.summarize_reviewed', [
    '--results-root', RUN, '--manifest', 'dataset/manifest.json',
    '--split', 'dataset/split.json', '--output', RUN/'reviewed_experiment_summary.json'], 300)
summary = json.loads((RUN/'reviewed_experiment_summary.json').read_text())
assert summary['split_labels_sources_groups_model_threshold_and_metrics_verified'] is True
print('Verified test accuracy:', summary['test_metrics']['accuracy'])
print('Verified candidate count:', summary['candidate_count'])


## Saved completed measurements

```json
{
  "evaluation": "single_locked_test_after_train_validation_model_selection",
  "partition_counts": {
    "train": {
      "recordings": 34,
      "groups": 34,
      "positive": 17,
      "negative": 17
    },
    "validation": {
      "recordings": 12,
      "groups": 12,
      "positive": 6,
      "negative": 6
    },
    "test": {
      "recordings": 12,
      "groups": 10,
      "positive": 6,
      "negative": 6
    }
  },
  "candidate_count": 52,
  "selected_candidate": {
    "representation": "semantic18",
    "head": "pooled",
    "classifier": "logistic",
    "preprocessing": "raw",
    "C": 0.01,
    "aggregation": "pooled"
  },
  "validation_metrics": {
    "n": 12,
    "accuracy": 0.8333333333333334,
    "balanced_accuracy": 0.8333333333333334,
    "precision": 0.8333333333333334,
    "recall": 0.8333333333333334,
    "specificity": 0.8333333333333334,
    "f1": 0.8333333333333334,
    "f2": 0.8333333333333334,
    "average_precision": 0.9055555555555554,
    "roc_auc": 0.8611111111111112,
    "confusion_matrix": [
      [
        5,
        1
      ],
      [
        1,
        5
      ]
    ],
    "confusion_order": [
      [
        "TN",
        "FP"
      ],
      [
        "FN",
        "TP"
      ]
    ],
    "localization": null,
    "mean_group_accuracy": 0.8333333333333334,
    "groups": 12,
    "accuracy_wilson_95pct": [
      0.5519691377470266,
      0.9530348578161463
    ],
    "confidence_interval_note": "Descriptive recording-level interval; related recording groups reduce effective sample size."
  },
  "test_metrics": {
    "n": 12,
    "accuracy": 0.4166666666666667,
    "balanced_accuracy": 0.41666666666666663,
    "precision": 0.4444444444444444,
    "recall": 0.6666666666666666,
    "specificity": 0.16666666666666666,
    "f1": 0.5333333333333333,
    "f2": 0.6060606060606061,
    "average_precision": 0.6510942760942761,
    "roc_auc": 0.5277777777777778,
    "confusion_matrix": [
      [
        1,
        5
      ],
      [
        2,
        4
      ]
    ],
    "confusion_order": [
      [
        "TN",
        "FP"
      ],
      [
        "FN",
        "TP"
      ]
    ],
    "localization": null,
    "mean_group_accuracy": 0.36666666666666664,
    "groups": 10,
    "accuracy_wilson_95pct": [
      0.19326031365875665,
      0.6804886874504503
    ],
    "confidence_interval_note": "Descriptive recording-level interval; related recording groups reduce effective sample size."
  },
  "test_domain_metrics": {
    "custom": {
      "n": 4,
      "accuracy": 0.5,
      "balanced_accuracy": 0.5,
      "precision": 0.5,
      "recall": 1.0,
      "specificity": 0.0,
      "f1": 0.6666666666666666,
      "f2": 0.8333333333333334,
      "average_precision": 1.0,
      "roc_auc": 1.0,
      "confusion_matrix": [
        [
          0,
          2
        ],
        [
          0,
          2
        ]
      ],
      "confusion_order": [
        [
          "TN",
          "FP"
        ],
        [
          "FN",
          "TP"
        ]
      ],
      "localization": null,
      "mean_group_accuracy": 0.3333333333333333,
      "groups": 2,
      "accuracy_wilson_95pct": [
        0.15003898915214947,
        0.8499610108478506
      ],
      "confidence_interval_note": "Descriptive recording-level interval; related recording groups reduce effective sample size."
    },
    "original": {
      "n": 8,
      "accuracy": 0.375,
      "balanced_accuracy": 0.375,
      "precision": 0.4,
      "recall": 0.5,
      "specificity": 0.25,
      "f1": 0.4444444444444444,
      "f2": 0.47619047619047616,
      "average_precision": 0.4928571428571429,
      "roc_auc": 0.375,
      "confusion_matrix": [
        [
          1,
          3
        ],
        [
          2,
          2
        ]
      ],
      "confusion_order": [
        [
          "TN",
          "FP"
        ],
        [
          "FN",
          "TP"
        ]
      ],
      "localization": null,
      "mean_group_accuracy": 0.375,
      "groups": 8,
      "accuracy_wilson_95pct": [
        0.13684428582359742,
        0.6942576053973727
      ],
      "confidence_interval_note": "Descriptive recording-level interval; related recording groups reduce effective sample size."
    }
  },
  "target_achieved_on_this_locked_test": false,
  "original_87pct_target_achieved_on_this_locked_test": false,
  "accuracy_above_80pct_on_new_sessions_established": false,
  "comparison_note": "Previous62.5%used40originalOOFdevelopment; this mixed12-recordtest is a different protocol and not a paired accuracy improvement."
}
```


## Export the reproducible result
Export trained heads, numeric reports, runtimes, exact source and the clean notebook. Private data, feature caches, audio, frames and workbooks remain excluded.

In [ ]:
# Generate the clean notebook against exact installed source bytes.
build_args = [sys.executable, 'tools/build_reviewed_notebook.py',
    '--baseline-cv', 'reference/clip_baseline_cv.json',
    '--notebook', str(WORKSPACE/'datathon_reviewed.ipynb'),
    '--bundle-out', str(RUN/'export_source.zip')]
measured_path = RUN/'reviewed_experiment_summary.json'
if measured_path.is_file():
    build_args += ['--measured-summary', str(measured_path)]
else:
    measured_reference = WORKSPACE/'reference/measured_reviewed_summary.json'
    if measured_reference.is_file():
        build_args += ['--measured-summary', str(measured_reference)]
subprocess.run(build_args, cwd=WORKSPACE, check=True)
members = {}
with zipfile.ZipFile(RUN/'export_source.zip') as source:
    for name in source.namelist():
        if name.endswith('.py'):
            assert source.read(name) == (WORKSPACE/name).read_bytes()
        members['source/'+name] = source.read(name)
for path in RUN.rglob('*'):
    if path.is_file() and path.suffix in {'.json', '.joblib', '.log'}:
        assert not (set(path.relative_to(WORKSPACE).parts) & {'features', 'dataset', 'open'})
        members[path.relative_to(WORKSPACE).as_posix()] = path.read_bytes()
for name in ('reviewed_experiment_summary.json', 'reviewed_model_comparison.csv',
             'reviewed_model_comparison.png'):
    path = WORKSPACE/'artifacts'/name
    if path.is_file():
        members['artifacts/'+name] = path.read_bytes()
notebook_path = WORKSPACE/'datathon_reviewed.ipynb'
clean = json.loads(notebook_path.read_text())
assert all(not cell.get('outputs') and cell.get('execution_count') is None
           for cell in clean['cells'] if cell['cell_type'] == 'code')
members[notebook_path.name] = notebook_path.read_bytes()
members['EXPORT_MANIFEST.json'] = json.dumps({
    'contains_media_workbooks_private_descriptions_or_feature_caches': False,
    'members_sha256': {name: hashlib.sha256(value).hexdigest()
                       for name,value in sorted(members.items())}}, indent=2).encode()
RESULTS_ZIP = BASE/'datathon_reviewed_results.zip'
with zipfile.ZipFile(RESULTS_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name,value in sorted(members.items()):
        archive.writestr(name,value)
print('Export:', RESULTS_ZIP, 'Bytes:', RESULTS_ZIP.stat().st_size)
print('SHA256:', hashlib.sha256(RESULTS_ZIP.read_bytes()).hexdigest())
if Path('/content').is_dir():
    from google.colab import files
    files.download(str(RESULTS_ZIP))
